# 🤖 Modelado y Evaluación con Modelo Base

Este notebook implementa el entrenamiento, optimización y evaluación de modelos de clasificación multiclase.

**Objetivo:** Entrenar y comparar múltiples algoritmos para seleccionar el mejor modelo.

## 1. Configuración e Importación de Librerías

In [0]:
# Celda 1: Recepción de Parámetros
schema_name = dbutils.widgets.text("schema_name", "")
schema_name = dbutils.widgets.get("schema_name")

silver_table = dbutils.widgets.text("silver_table", "")
silver_table = dbutils.widgets.get("silver_table")

In [0]:
from pyspark.ml import PipelineModel
from pyspark.ml.feature import VectorAssembler, StringIndexer, UnivariateFeatureSelector, OneHotEncoder, StandardScaler, PCA
from pyspark.sql import functions as F
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

## 2. Cargar el conjunto de `datos`

In [0]:
# generar nombre completo de la tabla
def qname(table):
    return f"{schema_name}.{table}"

SILVER_FULL = qname(silver_table)
print("Tabla Silver:", SILVER_FULL)

In [0]:
# Leer datos de la tabla Bronze
lpn_silver = spark.table(SILVER_FULL)

display(lpn_silver.limit(20))

In [0]:
# Dimensiones del dataset
print(f"Filas: {lpn_silver.count()}, Columnas: {len(lpn_silver.columns)}")

### 2.1 Division de datos en train y test

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

target_col = "LABEL_ZONE"
train_frac = 0.8

# Añadir columna aleatoria
df = lpn_silver.withColumn("rand", F.rand())

# Calcular percentil por clase
window = Window.partitionBy(target_col).orderBy("rand")
df = df.withColumn("row_number", F.row_number().over(window))
df = df.withColumn("count_per_class", F.count("*").over(Window.partitionBy(target_col)))
df = df.withColumn("frac", F.col("row_number") / F.col("count_per_class"))

# Split estratificado
train_df = df.filter(F.col("frac") <= train_frac).drop("rand", "row_number", "count_per_class", "frac")
test_df = df.filter(F.col("frac") > train_frac).drop("rand", "row_number", "count_per_class", "frac")

print(f"Train: {train_df.count()} filas")
print(f"Test:  {test_df.count()} filas")

## 3. Entramiento de modelo

In [0]:
# Carga el modelo desde la misma ruta
pipe = PipelineModel.load("/tmp/pipeline/transformer")
df_final = pipe.transform(train_df)

In [0]:
rf = RandomForestClassifier(featuresCol="features", labelCol="LABEL_ZONE_idx", seed=42, numTrees=100)
rf_model = rf.fit(df_final)

In [0]:
# Hacer predicciones en el conjunto de entrenamiento
train_predictions = rf_model.transform(df_final)

In [0]:
# Set correct label column for evaluator
evaluator = MulticlassClassificationEvaluator(
    predictionCol="prediction",
    labelCol="LABEL_ZONE_idx"
)

In [0]:
overallPrecision = evaluator.evaluate(train_predictions, {evaluator.metricName: "weightedPrecision"})
print("Overall Precision:", overallPrecision)
overallRecall = evaluator.evaluate(train_predictions, {evaluator.metricName: "weightedRecall"})
print("Overall Recall:", overallRecall)
overallF1 = evaluator.evaluate(train_predictions, {evaluator.metricName: "weightedFMeasure"})
print("Overall F1 Score:", overallF1)

## 4. Evaluacion de modelo

In [0]:
df_test = pipe.transform(test_df)

# Hacer predicciones en el conjunto de test
test_predictions = rf_model.transform(df_test)

In [0]:
overallPrecision = evaluator.evaluate(test_predictions, {evaluator.metricName: "weightedPrecision"})
print("Overall Precision:", overallPrecision)
overallRecall = evaluator.evaluate(test_predictions, {evaluator.metricName: "weightedRecall"})
print("Overall Recall:", overallRecall)
overallF1 = evaluator.evaluate(test_predictions, {evaluator.metricName: "weightedFMeasure"})
print("Overall F1 Score:", overallF1)